# Contrôle qualité des données - Échantillon stratifié Cifer Fraud Detection

À exécuter sur Cifer-echantillon-strat.csv, en complément de eda_echantillon.py.

Couvre :
  1. Valeurs manquantes
  2. Doublons
  3. Valeurs extrêmes / aberrantes (outliers génériques, hors lien avec la fraude)
  4. Cohérence des types et des plages de valeurs
  5. Cohérence logique métier (valeurs négatives, incohérences structurelles)

Note : l'analyse des outliers EN LIEN AVEC LA FRAUDE (taux de fraude dans
les outliers) reste dans eda_echantillon.py (axe 3), car c'est une analyse
exploratoire orientée métier, pas un contrôle qualité générique.
Ici, on regarde les outliers/anomalies indépendamment de la cible, comme
on le ferait sur n'importe quel dataset avant modélisation.

Auteur : Rasmané

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)
sns.set_theme(style="whitegrid")

DOSSIER_SORTIE = r"C:\Users\hp\Documents\Fraude_detection\data"

## 0. CHARGEMENT

In [ ]:
CHEMIN_FICHIER = r"C:\Users\hp\Documents\Fraude_detection\data\Cifer-echantillon-strat.csv"

DTYPES = {
    "step": "int16",
    "type": "category",
    "amount": "float32",
    "oldbalanceOrg": "float32",
    "newbalanceOrig": "float32",
    "oldbalanceDest": "float32",
    "newbalanceDest": "float32",
    "isFraud": "int8",
    "isFlaggedFraud": "int8",
}

data1 = pd.read_csv(CHEMIN_FICHIER, dtype=DTYPES)
print(f"Échantillon chargé : {data1.shape[0]:,} lignes, {data1.shape[1]} colonnes\n")

## 1. VALEURS MANQUANTES

In [ ]:
print("=" * 70)
print("1. VALEURS MANQUANTES")
print("=" * 70)

manquants = data1.isnull().sum()
pct_manquants = (manquants / len(data1) * 100).round(3)
tableau_manquants = pd.DataFrame({
    "nb_manquants": manquants,
    "pct_manquants": pct_manquants
}).sort_values("nb_manquants", ascending=False)

print(tableau_manquants)

if tableau_manquants["nb_manquants"].sum() == 0:
    print("\nAucune valeur manquante détectée sur l'échantillon.")
else:
    print(f"\nColonnes avec des manquants :")
    print(tableau_manquants[tableau_manquants["nb_manquants"] > 0])

    # Visualisation type "carte des manquants" si des colonnes sont concernées
    colonnes_avec_manquants = tableau_manquants[tableau_manquants["nb_manquants"] > 0].index.tolist()
    if colonnes_avec_manquants:
        plt.figure(figsize=(10, 5))
        sns.heatmap(data1[colonnes_avec_manquants].isnull(), cbar=False, cmap="Reds")
        plt.title("Carte des valeurs manquantes")
        plt.tight_layout()
        plt.savefig(f"{DOSSIER_SORTIE}/qc_valeurs_manquantes.png", dpi=120)
        #plt.close()
        print("Graphique sauvegardé : qc_valeurs_manquantes.png")

        # Manquants croisés avec isFraud : un manquant n'est jamais "aléatoire"
        # par hasard en fraude, toujours vérifier s'il est corrélé à la cible
        print("\nTaux de fraude selon présence de valeurs manquantes :")
        for col in colonnes_avec_manquants:
            masque_manquant = data1[col].isnull()
            taux_avec = data1.loc[masque_manquant, "isFraud"].mean()
            taux_sans = data1.loc[~masque_manquant, "isFraud"].mean()
            print(f"  {col} : manquant -> {taux_avec*100:.3f}% fraude | "
                  f"non-manquant -> {taux_sans*100:.3f}% fraude")

## 2. DOUBLONS

In [ ]:
print("\n" + "=" * 70)
print("2. DOUBLONS")
print("=" * 70)

# 2.1 Doublons stricts (toutes colonnes identiques)
nb_doublons_stricts = data1.duplicated().sum()
print(f"\n2.1 Doublons stricts (toutes colonnes) : {nb_doublons_stricts:,} "
      f"({nb_doublons_stricts/len(data1)*100:.4f}%)")

# 2.2 Doublons hors colonne technique fichier_source (si présente)
if "fichier_source" in data1.columns:
    nb_doublons_hors_source = data1.drop(columns=["fichier_source"]).duplicated().sum()
    print(f"2.2 Doublons hors 'fichier_source' : {nb_doublons_hors_source:,} "
          f"({nb_doublons_hors_source/len(data1)*100:.4f}%)")

# 2.3 Doublons sur les colonnes métier uniquement (hors identifiants,
# car nameOrig/nameDest sont uniques par construction sur données synthétiques)
colonnes_metier = ["step", "type", "amount", "oldbalanceOrg", "newbalanceOrig",
                    "oldbalanceDest", "newbalanceDest", "isFraud"]
colonnes_metier_presentes = [c for c in colonnes_metier if c in data1.columns]
nb_doublons_metier = data1.duplicated(subset=colonnes_metier_presentes).sum()
print(f"2.3 Doublons sur variables métier uniquement (hors ID) : {nb_doublons_metier:,} "
      f"({nb_doublons_metier/len(data1)*100:.4f}%)")
print("-> Un doublon ici peut signaler 2 transactions distinctes mais")
print("   parfaitement identiques en tous points (rare mais pas impossible,")
print("   à ne pas confondre avec un doublon technique de fusion).")

if nb_doublons_stricts > 0:
    print("\nAperçu de quelques lignes dupliquées :")
    print(data1[data1.duplicated(keep=False)].sort_values(
        by=colonnes_metier_presentes[0]).head(10))

## 3. VALEURS EXTRÊMES / ABERRANTES (génériques, hors lien avec la fraude)

In [ ]:
print("\n" + "=" * 70)
print("3. VALEURS EXTRÊMES ET ABERRANTES")
print("=" * 70)

colonnes_num = ["step", "amount", "oldbalanceOrg", "newbalanceOrig",
                 "oldbalanceDest", "newbalanceDest"]
colonnes_num = [c for c in colonnes_num if c in data1.columns]

# 3.1 Min / max / plage de valeurs (détecter des valeurs impossibles)
print("\n3.1 Plage de valeurs par colonne numérique :")
plage_valeurs = data1[colonnes_num].agg(["min", "max", "mean", "median", "std"]).T
print(plage_valeurs.round(2))

# 3.2 Valeurs négatives (impossibles pour montants/soldes)
print("\n3.2 Valeurs négatives détectées :")
for col in colonnes_num:
    nb_negatifs = (data1[col] < 0).sum()
    if nb_negatifs > 0:
        print(f"  {col} : {nb_negatifs:,} valeurs négatives "
              f"({nb_negatifs/len(data1)*100:.4f}%) -> ANOMALIE À INVESTIGUER")
    else:
        print(f"  {col} : aucune valeur négative (cohérent)")

# 3.3 Valeurs exactement nulles (0) - fréquence, pas anormal en soi mais à quantifier
print("\n3.3 Fréquence des valeurs à zéro :")
for col in colonnes_num:
    nb_zeros = (data1[col] == 0).sum()
    print(f"  {col} : {nb_zeros:,} zéros ({nb_zeros/len(data1)*100:.2f}%)")

# 3.4 Détection IQR générique (sans lien avec isFraud - vision "qualité pure")
print("\n3.4 Détection d'outliers via IQR (vision qualité, indépendante de isFraud) :")
resultats_iqr = []
for col in colonnes_num:
    q1, q3 = data1[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    borne_inf = q1 - 1.5 * iqr
    borne_sup = q3 + 1.5 * iqr
    nb_outliers = ((data1[col] < borne_inf) | (data1[col] > borne_sup)).sum()
    resultats_iqr.append({
        "variable": col,
        "borne_inf": round(borne_inf, 2),
        "borne_sup": round(borne_sup, 2),
        "nb_outliers": nb_outliers,
        "pct_outliers": round(nb_outliers / len(data1) * 100, 2)
    })
df_iqr = pd.DataFrame(resultats_iqr)
print(df_iqr)

# 3.5 Détection par z-score (méthode complémentaire, plus sensible aux extrêmes)
print("\n3.5 Détection d'outliers via z-score (|z| > 3) :")
resultats_zscore = []
for col in colonnes_num:
    moyenne = data1[col].mean()
    ecart_type = data1[col].std()
    z_scores = (data1[col] - moyenne) / ecart_type
    nb_extremes = (z_scores.abs() > 3).sum()
    resultats_zscore.append({
        "variable": col,
        "nb_extremes_zscore": nb_extremes,
        "pct_extremes_zscore": round(nb_extremes / len(data1) * 100, 2)
    })
df_zscore = pd.DataFrame(resultats_zscore)
print(df_zscore)
print("-> Le z-score est plus conservateur que l'IQR sur des distributions")
print("   très asymétriques (typique des montants financiers) ; comparer")
print("   les deux donne une idée de la sensibilité de chaque méthode ici.")

# 3.6 Visualisation boxplots génériques
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
for i, col in enumerate(colonnes_num):
    sns.boxplot(y=data1[col], ax=axes[i], color="#F4A261")
    axes[i].set_title(f"{col}")
for j in range(len(colonnes_num), len(axes)):
    fig.delaxes(axes[j])
plt.tight_layout()
plt.savefig(f"{DOSSIER_SORTIE}/qc_boxplots_generiques.png", dpi=120)
#plt.close()
print("\nGraphique sauvegardé : qc_boxplots_generiques.png")

# 3.7 step hors plage attendue (0-743 pour 744 steps / 30 jours)
if "step" in data1.columns:
    print("\n3.7 Vérification de la plage de 'step' (attendu : 0 à 743 inclus)")
    step_min, step_max = data1["step"].min(), data1["step"].max()
    print(f"  step min = {step_min}, step max = {step_max}")
    if step_min < 0 or step_max > 743:
        print("  ANOMALIE : step hors de la plage attendue (30 jours = 744 steps).")
    else:
        print("  Cohérent avec la période simulée de 30 jours.")

## 4. COHÉRENCE DES TYPES DE DONNÉES

In [ ]:
print("\n" + "=" * 70)
print("4. COHÉRENCE DES TYPES DE DONNÉES")
print("=" * 70)
print(data1.dtypes)

# Valeurs de 'type' effectivement présentes vs valeurs attendues du sujet
if "type" in data1.columns:
    types_attendus = {"CASH_IN", "CASH_OUT", "DEBIT", "PAYMENT", "TRANSFER"}
    types_presents = set(data1["type"].unique())
    print(f"\nTypes attendus (sujet) : {types_attendus}")
    print(f"Types présents (données) : {types_presents}")
    types_inattendus = types_presents - types_attendus
    types_manquants = types_attendus - types_presents
    if types_inattendus:
        print(f"ATTENTION - types inattendus trouvés : {types_inattendus}")
    if types_manquants:
        print(f"Types attendus mais absents de l'échantillon : {types_manquants}")
    if not types_inattendus and not types_manquants:
        print("Cohérent : tous les types attendus sont présents, aucun type inattendu.")

# isFraud et isFlaggedFraud : vérifier que ce sont bien des binaires 0/1
for col in ["isFraud", "isFlaggedFraud"]:
    if col in data1.columns:
        valeurs_uniques = sorted(data1[col].unique())
        statut = "OK (binaire 0/1)" if valeurs_uniques == [0, 1] else "ANOMALIE"
        print(f"\n{col} : valeurs uniques = {valeurs_uniques} -> {statut}")

## 5. COHÉRENCE LOGIQUE MÉTIER

In [ ]:
print("\n" + "=" * 70)
print("5. COHÉRENCE LOGIQUE MÉTIER")
print("=" * 70)

# 5.1 Transactions à montant nul
nb_montant_nul = (data1["amount"] == 0).sum()
print(f"\n5.1 Transactions avec montant = 0 : {nb_montant_nul:,} "
      f"({nb_montant_nul/len(data1)*100:.4f}%)")
if nb_montant_nul > 0:
    print(f"  Dont frauduleuses : {data1.loc[data1['amount']==0, 'isFraud'].sum()}")

# 5.2 Cohérence comptable émetteur : newbalanceOrig = oldbalanceOrg - amount
print("\n5.2 Cohérence comptable émetteur (écart |oldbalanceOrg - amount - newbalanceOrig|)")
data1["ecart_balance_orig"] = (
    data1["oldbalanceOrg"] - data1["amount"] - data1["newbalanceOrig"]
).abs()
tolerance = 1  # tolérance d'arrondi
nb_incoherent_orig = (data1["ecart_balance_orig"] > tolerance).sum()
print(f"  Transactions incohérentes (écart > {tolerance}) : {nb_incoherent_orig:,} "
      f"({nb_incoherent_orig/len(data1)*100:.2f}%)")
print(f"  Écart moyen : {data1['ecart_balance_orig'].mean():.2f}")
print(f"  Écart médian : {data1['ecart_balance_orig'].median():.2f}")
print(f"  Écart max : {data1['ecart_balance_orig'].max():.2f}")

# 5.3 Cohérence comptable destinataire : newbalanceDest = oldbalanceDest + amount
print("\n5.3 Cohérence comptable destinataire (écart |oldbalanceDest + amount - newbalanceDest|)")
data1["ecart_balance_dest"] = (
    data1["oldbalanceDest"] + data1["amount"] - data1["newbalanceDest"]
).abs()
nb_incoherent_dest = (data1["ecart_balance_dest"] > tolerance).sum()
print(f"  Transactions incohérentes (écart > {tolerance}) : {nb_incoherent_dest:,} "
      f"({nb_incoherent_dest/len(data1)*100:.2f}%)")
print(f"  Écart moyen : {data1['ecart_balance_dest'].mean():.2f}")
print(f"  Écart médian : {data1['ecart_balance_dest'].median():.2f}")

print("\nInterprétation : ces écarts confirment ou infirment la fidélité du")
print("simulateur Cifer par rapport à la logique comptable stricte de PaySim.")
print("Si le taux d'incohérence est élevé, documenter cette limite dans le")
print("rapport (section 'limites du projet' exigée par le sujet, point 19),")
print("et rester prudent sur l'usage de features dérivées de ces écarts.")

## 6. RÉCAPITULATIF FINAL

In [ ]:
print("\n" + "=" * 70)
print("6. RÉCAPITULATIF QUALITÉ DES DONNÉES")
print("=" * 70)
print(f"- Lignes analysées : {len(data1):,}")
print(f"- Valeurs manquantes totales : {int(tableau_manquants['nb_manquants'].sum())}")
print(f"- Doublons stricts : {nb_doublons_stricts:,}")
print(f"- Valeurs négatives détectées : "
      f"{sum((data1[c] < 0).sum() for c in colonnes_num)}")
print(f"- Incohérences comptables émetteur : {nb_incoherent_orig:,} "
      f"({nb_incoherent_orig/len(data1)*100:.2f}%)")
print(f"- Incohérences comptables destinataire : {nb_incoherent_dest:,} "
      f"({nb_incoherent_dest/len(data1)*100:.2f}%)")
print("\nContrôle qualité terminé — résultats prêts pour la section")
print("'qualité des données' du rapport et pour orienter le nettoyage")
print("avant le feature engineering.")